# This code was used to create the ChromaDB for semantic search and doesn't need to be run

In [ ]:
import bz2

# This first cell creates a list of articles from the simplewiki-latest-pages 

articles = []

with bz2.open(
    "simplewiki-latest-pages-articles.xml.bz2",
    "rt",
    encoding="utf-8"
) as f:

    current_title = None
    current_text = []

    for line in f:

        # Article titles start/end with "="
        if line.startswith("=") and line.endswith("=\n"):
            
            # save previous article
            if current_title:
                articles.append({
                    "title": current_title,
                    "text": "".join(current_text)
                })

            current_title = line.strip("=\n ").strip()
            current_text = []

        else:
            current_text.append(line)

# add final article
if current_title:
    articles.append({
        "title": current_title,
        "text": "".join(current_text)
    })

len(articles)

723542

In [ ]:
# This cell filters the list of articles to focus only on the science articles that are of sufficient length

science_keywords = ['physics', 'chemistry', 'biology', 'astronomy', 
                    'molecule', 'atom', 'planet', 'evolution', 'cell',
                    'element', 'energy', 'force', 'species', 'genome']

science_articles = [a for a in articles 
                    if any(kw in a['title'].lower() for kw in science_keywords)]

science_and_long = [a for a in articles 
                    if any(kw in a['title'].lower() for kw in science_keywords)
                    and len(a['text']) >= 200]

In [ ]:
# This cell cleans the articles to make sure that they can be passed easily for the embedding process 

import re
import html

def clean_text(text: str) -> str:
    """Clean noisy text while preserving readable content."""
    if not isinstance(text, str):
        return text

    # [[Page|display text]] → keep display text
    text = re.sub(r'\[\[[^\]|]+\|([^\]]+)\]\]', r'\1', text)
    # [[Page]] → keep page text  
    text = re.sub(r'\[\[([^\]]+)\]\]', r'\1', text) 

    # Decode HTML entities (&amp;, &nbsp;, etc.)
    text = html.unescape(text)

    # Remove citation markers like [1], [citation needed]
    text = re.sub(r'\[[^\]]*\]', '', text)

    # Remove parenthetical pronunciations
    text = re.sub(r'\([^)]*[/\\][^)]*\)', '', text)

    # Replace newlines/tabs with spaces
    text = re.sub(r'[\r\n\t]+', ' ', text)

    # Remove non-printable characters
    text = ''.join(c for c in text if c.isprintable())

    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove spaces before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    return text.strip()


def clean_articles(articles):
    """
    Clean all string values in a list of article dictionaries.
    """
    cleaned_articles = []

    for article in articles:
        cleaned_article = {
            key: clean_text(value) if isinstance(value, str) else value
            for key, value in article.items()
        }
        cleaned_articles.append(cleaned_article)

    return cleaned_articles

In [30]:
cleaned_articles = clean_articles(science_and_long)

In [28]:
from openai import OpenAI
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('/Users/Greg/Desktop/DSI/deploying-ai/05_src/.secrets')

openai_client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)


def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

In [37]:
import chromadb

chroma_client = chromadb.PersistentClient()
chroma_client.delete_collection("science_facts")
collection = chroma_client.get_or_create_collection(name="science_facts")

In [36]:
BATCH_SIZE = 25
MAX_CHARS = 8000

for i in range(0, len(cleaned_articles), BATCH_SIZE):
    batch = cleaned_articles[i:i + BATCH_SIZE]
    
    texts = [a['text'][:MAX_CHARS] for a in batch]
    ids = [f"doc_{i+j}" for j in range(len(batch))]
    metadatas = [{"title": a['title']} for a in batch]
    
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    embeddings = [item.embedding for item in response.data]
    
    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas
    )
    
    print(f"Processed {min(i + BATCH_SIZE, len(cleaned_articles))}/{len(cleaned_articles)}")

print("Items in collection:", collection.count())

Processed 25/2424
Processed 50/2424
Processed 75/2424
Processed 100/2424
Processed 125/2424
Processed 150/2424
Processed 175/2424
Processed 200/2424
Processed 225/2424
Processed 250/2424
Processed 275/2424
Processed 300/2424
Processed 325/2424
Processed 350/2424
Processed 375/2424
Processed 400/2424
Processed 425/2424
Processed 450/2424
Processed 475/2424
Processed 500/2424
Processed 525/2424
Processed 550/2424
Processed 575/2424
Processed 600/2424
Processed 625/2424
Processed 650/2424
Processed 675/2424
Processed 700/2424
Processed 725/2424
Processed 750/2424
Processed 775/2424
Processed 800/2424
Processed 825/2424
Processed 850/2424
Processed 875/2424
Processed 900/2424
Processed 925/2424
Processed 950/2424
Processed 975/2424
Processed 1000/2424
Processed 1025/2424
Processed 1050/2424
Processed 1075/2424
Processed 1100/2424
Processed 1125/2424
Processed 1150/2424
Processed 1175/2424
Processed 1200/2424
Processed 1225/2424
Processed 1250/2424
Processed 1275/2424
Processed 1300/2424
Pr

In [38]:
print(client.list_collections())

[Collection(name=science_facts)]
